# AI-Based College Academic Assistant – RAG Module

## College Document Retrieval System

This module implements the Retrieval-Augmented Generation (RAG)
component of the AI-Based College Academic Assistant.

### Objectives
- Load college documents
- Extract text from PDFs
- Split documents into chunks
- Generate embeddings
- Store embeddings in a vector database
- Retrieve relevant document chunks
- Generate context-based answers using an LLM
- Test document retrieval and responses

In [ ]:
!pip install -q langchain langchain-community langchain-openai \
pypdf chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/

In [ ]:
from google.colab import files

uploaded = files.upload()


Saving B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf to B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
Saving B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf to B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf


In [ ]:
import os

print("Files uploaded:")
for file in os.listdir():
    if file.endswith(".pdf"):
        print("-", file)

Files uploaded:
- B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
- B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import glob
import os

# Find every PDF uploaded to Colab
pdf_files = glob.glob("/content/*.pdf")

print("PDF files found:")
for pdf in pdf_files:
    print("-", os.path.basename(pdf))

documents = []

for pdf in pdf_files:
    print(f"\nLoading: {os.path.basename(pdf)}")

    loader = PyPDFLoader(pdf)
    docs = loader.load()

    documents.extend(docs)

    print(f"Pages loaded: {len(docs)}")

print("\n✅ Total pages loaded:", len(documents))

/tmp/ipykernel_989/1231253492.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF files found:
- B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
- B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf

Loading: B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
Pages loaded: 111

Loading: B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
Pages loaded: 152

✅ Total pages loaded: 263


In [ ]:
# Inspect the first document

print("SOURCE:")
print(documents[0].metadata)

print("\nTEXT:")
print(documents[0].page_content[:2000])

SOURCE:
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-08-28T12:45:17-05:00', 'author': 'user', 'moddate': '2026-08-28T12:45:17-05:00', 'source': '/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf', 'total_pages': 111, 'page': 0, 'page_label': '1'}

TEXT:
(Established under Section 3 of UGC Act, 1956) 
Placed under Category ‘A’ by MHRD, GoI | Accredited with ‘A+’      
Grade by NAAC 
 
Regulations and Curriculum for 
Bachelor of Technology (B.Tech.) 
in 
Electrical and Electronics Engineering 
Electronics and Communication Engineering 
Electronics Engineering (VLSI Design Technology) 
Electronics and Communication (Advanced 
Communication Technology) 
Version 2026.01


In [ ]:
# Find the first page belonging to the ISE regulations PDF

for doc in documents:
    if "B.Tech._IS_2024-01_Regulations_Syllabus.pdf" in doc.metadata["source"]:
        print("SOURCE:")
        print(doc.metadata)

        print("\nTEXT:")
        print(doc.page_content[:2000])
        break

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print("Total original pages:", len(documents))
print("Total chunks created:", len(chunks))

Total original pages: 263
Total chunks created: 752


In [ ]:
# Inspect a few chunks

for i in range(3):
    print("=" * 80)
    print(f"CHUNK {i + 1}")
    print("=" * 80)
    print(chunks[i].page_content[:1500])
    print("\nMETADATA:")
    print(chunks[i].metadata)
    print()

CHUNK 1
(Established under Section 3 of UGC Act, 1956) 
Placed under Category ‘A’ by MHRD, GoI | Accredited with ‘A+’      
Grade by NAAC 
 
Regulations and Curriculum for 
Bachelor of Technology (B.Tech.) 
in 
Electrical and Electronics Engineering 
Electronics and Communication Engineering 
Electronics Engineering (VLSI Design Technology) 
Electronics and Communication (Advanced 
Communication Technology) 
Version 2026.01

METADATA:
{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-08-28T12:45:17-05:00', 'author': 'user', 'moddate': '2026-08-28T12:45:17-05:00', 'source': '/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf', 'total_pages': 111, 'page': 0, 'page_label': '1'}

CHUNK 2
VISION 
 
To build a humane society through excellence in education and healthcare 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
MISSION 
 
To develop 
Nitte (Deemed to be University) 
As a center of excellence imparting quality education, 
Generating competent, skilled manpower to 

In [ ]:
# Search for chunks containing "ATTENDANCE REQUIREMENT"

attendance_chunks = []

for chunk in chunks:
    if "ATTENDANCE REQUIREMENT" in chunk.page_content.upper():
        attendance_chunks.append(chunk)

print("Attendance-related chunks found:", len(attendance_chunks))

for i, chunk in enumerate(attendance_chunks[:3]):
    print("\n" + "=" * 80)
    print(f"ATTENDANCE CHUNK {i + 1}")
    print("=" * 80)
    print(chunk.page_content)
    print("\nSOURCE:", chunk.metadata.get("source"))
    print("PAGE:", chunk.metadata.get("page_label"))

Attendance-related chunks found: 7

ATTENDANCE CHUNK 1
CONTENTS 
 
1. INTRODUCTION 1 
2. ELIGIBILITY FOR ADMISSION 2 
3. PROGRAM PATHS, EXIT OPTIONS, AND DURATION OF THE B. TECH. PROGRAM 3 
4. DEGREE PROGRAMS 5 
5. CREDIT SYSTEM 5 
6. REGISTRATION 7 
7. ADD/DROP/AUDIT OPTIONS 8 
8. COURSE STRUCTURE 8 
9. ATTENDANCE REQUIREMENT 14 
10. WITHDRAWAL FROM THE PROGRAM 15 
11. EVALUATION SYSTEM 16 
12. EVALUATION OF PERFORMANCE 22 
13. COMMUNICATION OF GRADES 22 
14. REQUIREMENTS FOR VERTICAL PROGRESSION 22 
15. AWARD OF CLASS 23 
16. APPEAL FOR REVIEW OF GRADES 23 
17. AWARD OF DEGREE 23 
18. GRADUATION REQUIREMENTS AND CONVOCATION 28 
19. AWARD OF PRIZES, MEDALS, CLASS & RANKS 28 
20. CONDUCT AND DISCIPLINE 29 
21. Pursuing Two Academic Programs 30 
APPENDIX - A 31 
APPENDIX-B 35

SOURCE: /content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
PAGE: 9

ATTENDANCE CHUNK 2
A student can register for courses for audit only, to supplement his/her knowledge and/or 
skills. The audit courses shall not be c

In [ ]:
!pip install -q langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [ ]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./college_vector_db"
)

print("✅ Vector database created!")
print("Total chunks stored:", len(chunks))

✅ Vector database created!
Total chunks stored: 752


In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("✅ Retriever created!")

✅ Retriever created!


In [ ]:
query = "What is the minimum attendance required for students?"

retrieved_docs = retriever.invoke(query)

print("🔎 Query:", query)
print("\nNumber of documents retrieved:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print("\n" + "=" * 80)
    print(f"RETRIEVED DOCUMENT {i}")
    print("=" * 80)
    print(doc.page_content[:1000])
    print("\n📄 Source:", doc.metadata.get("source"))
    print("📌 Page:", doc.metadata.get("page_label"))

🔎 Query: What is the minimum attendance required for students?

Number of documents retrieved: 4

RETRIEVED DOCUMENT 1
commencing from the 5th semester. 
8.6.2 The minimum number of students to be registered for any Elective offered shall not 
be less than fifteen (15) and should not exceed forty (40). 
8.6.3 A candidate shall opt for his/her choice of electives and register for the same at the 
beginning of each of the 5th to 7th semesters if pre -registration is not done. The 
candidate is permitted to opt for a change of elective within 15 days from the date 
of commencement of the semester as per the academic calendar of the college. 
 
9. ATTENDANCE REQUIREMENT 
9.1 Each semester is considered as a unit and the candidate has to put in a minimum attendance 
of 85% in each subject with a provision of condoning 10% of the attendance by the 
principal for reasons such as medical grounds, participation in University level sports, 
cultural activities, seminars, workshops, and paper pre

In [ ]:
query = "What are the rules for changing an elective?"

retrieved_docs = retriever.invoke(query)

print("🔎 Query:", query)
print("\nNumber of documents retrieved:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print("\n" + "=" * 80)
    print(f"RETRIEVED DOCUMENT {i}")
    print("=" * 80)
    print(doc.page_content[:800])
    print("\n📄 Source:", doc.metadata.get("source"))
    print("📌 Page:", doc.metadata.get("page_label"))

🔎 Query: What are the rules for changing an elective?

Number of documents retrieved: 4

RETRIEVED DOCUMENT 1
commencing from the 5th semester. 
8.6.2 The minimum number of students to be registered for any Elective offered shall not 
be less than fifteen (15) and should not exceed forty (40). 
8.6.3 A candidate shall opt for his/her choice of electives and register for the same at the 
beginning of each of the 5th to 7th semesters if pre -registration is not done. The 
candidate is permitted to opt for a change of elective within 15 days from the date 
of commencement of the semester as per the academic calendar of the college. 
 
9. ATTENDANCE REQUIREMENT 
9.1 Each semester is considered as a unit and the candidate has to put in a minimum attendance 
of 85% in each subject with a provision of condoning 10% of the attendance by the 
principal for reasons such as medical grounds, participation in 

📄 Source: /content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
📌 Page: 26

RETRIEVED DOCUMENT 2

## RAG Retrieval Testing

The retriever searches the college document vector database for information relevant to a student's query.

Example queries tested:
1. What is the minimum attendance required for students?
2. What are the rules for changing an elective?

The system successfully retrieved relevant sections from the uploaded college regulations and syllabus documents, along with source file and page metadata.

In [ ]:
!pip install -q langchain-openai

In [ ]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

GROQ_API_KEY = "#####GROQ API KEY"

if GROQ_API_KEY:
    print("✅ Groq API key loaded securely!")
else:
    print("❌ Groq API key not found.")

✅ Groq API key loaded securely!


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0
)

print("✅ Groq LLM connected!")

✅ Groq LLM connected!


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are an AI-based College Academic Assistant.

Answer the student's question using ONLY the information contained
in the retrieved college documents.

Rules:
1. Do not use outside knowledge.
2. Do not invent or assume information.
3. If the retrieved documents do not contain enough information,
   say:
   "I couldn't find enough information about this in the available
   college documents."
4. Do not claim that the retrieved information is complete unless
   the documents explicitly say that it is complete.
5. Preserve important numbers, dates, conditions, and terminology
   exactly as stated in the documents.
6. Give a clear and concise answer.
7. Mention the relevant source document and page when appropriate.

Retrieved College Documents:
{context}

Student Question:
{question}

Answer:
""")

print("✅ Improved RAG prompt created!")

✅ Improved RAG prompt created!


In [ ]:
def ask_college_assistant(question):
    # Retrieve relevant college documents
    retrieved_docs = retriever.invoke(question)

    # Prepare the retrieved context
    context_parts = []

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown source")
        page = doc.metadata.get("page_label", "Unknown page")

        context_parts.append(
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content:\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # Create the prompt
    messages = rag_prompt.format_messages(
        context=context,
        question=question
    )

    # Generate answer using Groq
    response = llm.invoke(messages)

    return response.content, retrieved_docs


print("✅ RAG pipeline created!")

✅ RAG pipeline created!


In [ ]:
question = "What is the minimum attendance required for students?"

answer, retrieved_docs = ask_college_assistant(question)

print("👩‍🎓 Student Question:")
print(question)

print("\n🤖 AI College Assistant:")
print(answer)

👩‍🎓 Student Question:
What is the minimum attendance required for students?

🤖 AI College Assistant:
The college requires each student to maintain **a minimum attendance of 85 % in every subject each semester** (with up to 10 % attendance condoned by the principal for valid reasons).  

**Source:** B.Tech EEE/ECE/VLSI, ACT 1st Year, page 26 (Attendance Requirement 9.1).


In [ ]:
print("📚 SOURCES USED BY THE AI")
print("=" * 80)

for i, doc in enumerate(retrieved_docs, 1):
    source = doc.metadata.get("source", "Unknown")
    page = doc.metadata.get("page_label", "Unknown")

    print(f"\n{i}. 📄 {source}")
    print(f"   📌 Page: {page}")

📚 SOURCES USED BY THE AI

1. 📄 /content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
   📌 Page: 26

2. 📄 /content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
   📌 Page: 32

3. 📄 /content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
   📌 Page: 32

4. 📄 /content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
   📌 Page: 26


In [ ]:
question = "What is the current weather in Mangaluru?"

answer, retrieved_docs = ask_college_assistant(question)

print("👩‍🎓 Student Question:")
print(question)

print("\n🤖 AI College Assistant:")
print(answer)

👩‍🎓 Student Question:
What is the current weather in Mangaluru?

🤖 AI College Assistant:
I couldn't find enough information about this in the available college documents.


In [ ]:
question = "What are the rules regarding attendance and changing electives?"

answer, retrieved_docs = ask_college_assistant(question)

print("👩‍🎓 Student Question:")
print(question)

print("\n🤖 AI College Assistant:")
print(answer)

print("\n📚 SOURCES USED:")
print("=" * 80)

for i, doc in enumerate(retrieved_docs, 1):
    source = doc.metadata.get("source", "Unknown")
    page = doc.metadata.get("page_label", "Unknown")

    print(f"{i}. {source}")
    print(f"   Page: {page}")

👩‍🎓 Student Question:
What are the rules regarding attendance and changing electives?

🤖 AI College Assistant:
**Elective‑change rules**

- A student may change the elective he/she has chosen **within 15 days from the date of commencement of the semester** (as given in the academic calendar).【/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf, p. 26】【/content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf, p. 32】  
- An elective must have **at least 15 and at most 40 students registered**.【/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf, p. 26】

**Attendance rules**

1. **Minimum attendance** – A student must maintain **at least 85 % attendance in each subject** for the semester.【/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf, p. 26】【/content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf, p. 32】

2. **Condonation** – The principal may **condone up to 10 % of the attendance** for reasons such as medical grounds, participation in university‑level sports, cultural activities, seminars, workshops, and paper pres

In [ ]:
def run_rag_demo(question):
    answer, retrieved_docs = ask_college_assistant(question)

    print("👩‍🎓 Student Question:")
    print(question)

    print("\n🤖 AI College Assistant:")
    print(answer)

    print("\n📚 SOURCES USED:")
    print("=" * 80)

    seen_sources = set()

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page_label", "Unknown")

        source_name = source.split("/")[-1]
        source_key = (source_name, page)

        if source_key not in seen_sources:
            print(f"📄 {source_name}")
            print(f"   📌 Page: {page}\n")
            seen_sources.add(source_key)


print("✅ Clean RAG demo function created!")

✅ Clean RAG demo function created!


In [ ]:
run_rag_demo("What is the minimum attendance required?")

👩‍🎓 Student Question:
What is the minimum attendance required?

🤖 AI College Assistant:
The college requires students to maintain a **minimum attendance of 85 % in each subject**.  

*Source: B.Tech.EEE, ECE, VLSI, ACT 1st Year – page 26 (Attendance Requirement 9.1); also B.Tech.AER, MEC, CIV, BTY 1st Year – page 32 (Attendance Requirement 9.1).*

📚 SOURCES USED:
📄 B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
   📌 Page: 26

📄 B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
   📌 Page: 32



In [ ]:
run_rag_demo("What are the rules for changing an elective?")

👩‍🎓 Student Question:
What are the rules for changing an elective?

🤖 AI College Assistant:
The college rules allow a student to change an elective **only within the first 15 days after the semester begins** (as set out in the academic calendar).

- Electives are taken from the 5th semester onward.  
- If a student has not pre‑registered, he/she must register for the chosen elective at the beginning of the 5th, 6th or 7th semester.  
- After registration, a change of elective is permitted **within 15 days from the date of commencement of the semester**.

**Source:** Both the B.Tech EEE/ECE/VLSI document (page 26) and the B.Tech AER/MEC/CIV/BTY document (page 32) state the 15‑day change window and the registration timing.

📚 SOURCES USED:
📄 B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
   📌 Page: 26

📄 B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
   📌 Page: 32

📄 B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
   📌 Page: 131

📄 B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
   📌 Page: 13



In [ ]:
test_questions = [
    "What is the minimum attendance required for students?",
    "What are the rules for changing an elective?",
    "What happens if a student has less than 75% attendance?",
    "What information is available about the 5th semester?",
    "What is the current weather in Mangaluru?"
]

print("🧪 RAG TEST QUESTIONS")
print("=" * 80)

for i, question in enumerate(test_questions, 1):
    print(f"{i}. {question}")

🧪 RAG TEST QUESTIONS
1. What is the minimum attendance required for students?
2. What are the rules for changing an elective?
3. What happens if a student has less than 75% attendance?
4. What information is available about the 5th semester?
5. What is the current weather in Mangaluru?


In [ ]:
for i, question in enumerate(test_questions, 1):

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)

    answer, retrieved_docs = ask_college_assistant(question)

    print("\n👩‍🎓 Question:")
    print(question)

    print("\n🤖 Answer:")
    print(answer)

    print("\n📚 Sources:")

    seen_sources = set()

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page_label", "Unknown")
        source_name = source.split("/")[-1]

        source_key = (source_name, page)

        if source_key not in seen_sources:
            print(f"- {source_name} | Page {page}")
            seen_sources.add(source_key)


TEST 1

👩‍🎓 Question:
What is the minimum attendance required for students?

🤖 Answer:
The college requires each student to maintain **a minimum attendance of 85 % in every subject each semester**【/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf, p. 26】.

📚 Sources:
- B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf | Page 26
- B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf | Page 32

TEST 2

👩‍🎓 Question:
What are the rules for changing an elective?

🤖 Answer:
The college rules allow a student to change an elective **only within the first 15 days after the semester begins** (as set out in the academic calendar).

- Electives are taken from the 5th semester onward.  
- If a student has not pre‑registered, he/she must register for the chosen elective at the beginning of the 5th, 6th or 7th semester.  
- After registration, a change of elective is permitted **within 15 days from the date of commencement of the semester**.

**Source:** Both the B.Tech EEE/ECE/VLSI document (page 26) and the B.Tech AER/MEC/CIV/

In [ ]:
results_with_scores = vector_db.similarity_search_with_score(
    "What is the current weather in Mangaluru?",
    k=4
)

print("🔎 Retrieval scores for an unrelated question")
print("=" * 80)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n{i}. Score: {score:.4f}")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Page: {doc.metadata.get('page_label')}")
    print(f"Text: {doc.page_content[:300]}...")

🔎 Retrieval scores for an unrelated question

1. Score: 1.4113
Source: /content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
Page: 148
Text: Regulations and Syllabus - B. Tech.  
 
Page | 136   
● Parasuraman retrieved a lot of lands and gave it to Indians. A Pandya king called Nilam Tharu Vil Nediyon 
built sea walls to prevent the sea from invading the land.  
● Balraman always travelled with an axe to clear the forests and make them c...

2. Score: 1.4760
Source: /content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
Page: 8
Text: 2. Nitte Meenakshi Institute of Technology 
N(DU) Off -Campus Centre, Bangalore 
            P.B.No.6429, Yelahanka, Bangalore - 560064 
Grievance and 
dispute resolution 
All disputes arising from this set of regulations shall be addressed to 
the Executive Council. The decision of the Executive Co...

3. Score: 1.4764
Source: /content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
Page: 77
Text: Third Edition, 2015. 
5. Erach Bharucha ―Textbook of Environmental Studies for Under

In [ ]:
results_with_scores = vector_db.similarity_search_with_score(
    "What is the minimum attendance required for students?",
    k=4
)

print("🔎 Retrieval scores for a relevant question")
print("=" * 80)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n{i}. Score: {score:.4f}")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Page: {doc.metadata.get('page_label')}")
    print(f"Text: {doc.page_content[:300]}...")

🔎 Retrieval scores for a relevant question

1. Score: 0.6406
Source: /content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf
Page: 26
Text: commencing from the 5th semester. 
8.6.2 The minimum number of students to be registered for any Elective offered shall not 
be less than fifteen (15) and should not exceed forty (40). 
8.6.3 A candidate shall opt for his/her choice of electives and register for the same at the 
beginning of each of...

2. Score: 0.6413
Source: /content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
Page: 32
Text: candidate is permitted to opt for a change of elective within 15 days from the date of 
commencement of the semester as per the academic calendar of the college. 
 
9. ATTENDANCE REQUIREMENT 
9.1 Each semester is considered as a unit and the candidate has to put in a minimum 
attendance of 85% in ea...

3. Score: 0.6465
Source: /content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf
Page: 32
Text: 9.3 The students shall be informed about their attendance position in the first week

In [ ]:
test_score_queries = [
    "What are the rules for changing an elective?",
    "Who is the Prime Minister of India?"
]

for query in test_score_queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = vector_db.similarity_search_with_score(query, k=4)

    for i, (doc, score) in enumerate(results, 1):
        print(
            f"{i}. Score: {score:.4f} | "
            f"Page: {doc.metadata.get('page_label')} | "
            f"{doc.page_content[:120].replace(chr(10), ' ')}..."
        )


QUERY: What are the rules for changing an elective?
1. Score: 1.1373 | Page: 26 | commencing from the 5th semester.  8.6.2 The minimum number of students to be registered for any Elective offered shall ...
2. Score: 1.2124 | Page: 32 | 8.5 Capstone Project Work/Professional Practice Report   8.5.1 Project work can be carried out as domain -specific /inte...
3. Score: 1.3566 | Page: 131 | Regulations and Syllabus - B. Tech.     Page | 119    Course Outcomes  At the end of the course student will be able to ...
4. Score: 1.3647 | Page: 13 | shall supersede all the corresponding earlier sets  of regulations of the B. Tech Degree  program of NITTE (Deemed to be...

QUERY: Who is the Prime Minister of India?
1. Score: 1.1507 | Page: 108 | Cabinet Mission Plan of 1946 – Partition Plan of 1947 and Indian Independence Act 1947, Constituent  Assembly and Commit...
2. Score: 1.1507 | Page: 136 | Cabinet Mission Plan of 1946 – Partition Plan of 1947 and Indian Independence Act 1947, Constituent 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

relevance_prompt = ChatPromptTemplate.from_template("""
You are checking whether retrieved college documents contain
enough information to answer a student's question.

Student Question:
{question}

Retrieved Documents:
{context}

Reply with ONLY one word:

YES
if the retrieved documents contain enough relevant information
to answer the question.

NO
if the documents are unrelated or do not contain enough
information to answer the question.

Do not use outside knowledge.
""")

print("✅ Relevance-check prompt created!")

✅ Relevance-check prompt created!


In [ ]:
question = "What is the minimum attendance required for students?"

retrieved_docs = retriever.invoke(question)

context = "\n\n---\n\n".join(
    doc.page_content for doc in retrieved_docs
)

messages = relevance_prompt.format_messages(
    question=question,
    context=context
)

relevance_response = llm.invoke(messages)

print("Question:", question)
print("Relevance check:", relevance_response.content.strip())

Question: What is the minimum attendance required for students?
Relevance check: YES


In [ ]:
question = "What is the current weather in Mangaluru?"

retrieved_docs = retriever.invoke(question)

context = "\n\n---\n\n".join(
    doc.page_content for doc in retrieved_docs
)

messages = relevance_prompt.format_messages(
    question=question,
    context=context
)

relevance_response = llm.invoke(messages)

print("Question:", question)
print("Relevance check:", relevance_response.content.strip())

Question: What is the current weather in Mangaluru?
Relevance check: NO


In [ ]:
def ask_college_assistant(question):

    # 1. Retrieve relevant college documents
    retrieved_docs = retriever.invoke(question)

    # 2. Prepare context for relevance checking
    context = "\n\n---\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # 3. Check whether retrieved documents are relevant
    relevance_messages = relevance_prompt.format_messages(
        question=question,
        context=context
    )

    relevance_response = llm.invoke(relevance_messages)
    relevance = relevance_response.content.strip().upper()

    # 4. If documents are not relevant, don't generate an answer
    if "YES" not in relevance:
        return (
            "I couldn't find enough information about this "
            "in the available college documents.",
            retrieved_docs
        )

    # 5. Prepare detailed context with source and page
    context_parts = []

    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown source")
        page = doc.metadata.get("page_label", "Unknown page")

        context_parts.append(
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content:\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # 6. Generate answer using the RAG prompt
    messages = rag_prompt.format_messages(
        context=context,
        question=question
    )

    response = llm.invoke(messages)

    return response.content, retrieved_docs


print("✅ RAG pipeline updated with relevance checking!")

✅ RAG pipeline updated with relevance checking!


In [ ]:
test_queries = [
    "What is the minimum attendance required for students?",
    "What is the current weather in Mangaluru?"
]

for i, question in enumerate(test_queries, 1):

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)

    answer, retrieved_docs = ask_college_assistant(question)

    print("\n👩‍🎓 Question:")
    print(question)

    print("\n🤖 Answer:")
    print(answer)


TEST 1

👩‍🎓 Question:
What is the minimum attendance required for students?

🤖 Answer:
The college requires each student to maintain **a minimum attendance of 85 % in every subject each semester**【/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf, p. 26】.

TEST 2

👩‍🎓 Question:
What is the current weather in Mangaluru?

🤖 Answer:
I couldn't find enough information about this in the available college documents.


In [ ]:
question = "What is the minimum attendance required for students?"

# A. Basic LLM without RAG
basic_response = llm.invoke(
    f"""
Answer the following question directly.

Question:
{question}
"""
)

# B. RAG-based assistant
rag_answer, _ = ask_college_assistant(question)

print("=" * 80)
print("A. BASIC LLM — WITHOUT RAG")
print("=" * 80)
print(basic_response.content)

print("\n" + "=" * 80)
print("B. COLLEGE RAG ASSISTANT")
print("=" * 80)
print(rag_answer)

A. BASIC LLM — WITHOUT RAG
The minimum attendance required for students is typically **75 % of the total classes**.

B. COLLEGE RAG ASSISTANT
The college requires each student to maintain **a minimum attendance of 85 % in every subject each semester**【/content/B.Tech.EEE,ECE,VLSI,ACT 1st Year.pdf, p. 26】.


## Basic LLM vs RAG Comparison

The same college-specific question was given to both a basic LLM and the RAG-based College Academic Assistant.

### Question
"What is the minimum attendance required for students?"

### Basic LLM
The basic LLM provided a general attendance value and stated that requirements vary by institution.

### RAG-based College Academic Assistant
The RAG system retrieved the relevant attendance requirement from the uploaded college regulations and answered that the minimum attendance is 85% in each subject.

### Observation
The comparison demonstrates that RAG can provide institution-specific answers by grounding the LLM response in the college's uploaded documents and associated source/page information.

In [ ]:
print("Retriever:", retriever)
print("Vector DB:", vector_db)

Retriever: tags=['Chroma', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7ec3000b4c20> search_kwargs={'k': 4}
Vector DB: <langchain_community.vectorstores.chroma.Chroma object at 0x7ec3000b4c20>


In [ ]:
test_docs = retriever.invoke(
    "What are the attendance requirements?"
)

print(f"Retrieved {len(test_docs)} documents\n")

for i, doc in enumerate(test_docs):
    print(f"--- Document {i+1} ---")
    print(doc.page_content[:500])
    print("Source:", doc.metadata.get("source"))
    print()

Retrieved 4 documents

--- Document 1 ---
candidate is permitted to opt for a change of elective within 15 days from the date of 
commencement of the semester as per the academic calendar of the college. 
 
9. ATTENDANCE REQUIREMENT 
9.1 Each semester is considered as a unit and the candidate has to put in a minimum 
attendance of 85% in each subject with a provision of condoning 10% of the attendance 
by the principal for reasons such as medical grounds, participation in  University level 
sports, cultural activities, seminars, worksho
Source: /content/B.Tech.AER, MEC, CIV, BTY 1ST YEAR.pdf

--- Document 2 ---
commencing from the 5th semester. 
8.6.2 The minimum number of students to be registered for any Elective offered shall not 
be less than fifteen (15) and should not exceed forty (40). 
8.6.3 A candidate shall opt for his/her choice of electives and register for the same at the 
beginning of each of the 5th to 7th semesters if pre -registration is not done. The 
candidate is per

In [ ]:
!pip install -q langgraph langchain-core langchain-groq

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Study_planner.py to Study_planner.py
Saving planner.py to planner.py
Saving graph.py to graph.py


In [1]:
import os

print(os.listdir())

['.config', 'sample_data']
